In [1]:
# Data manipulation
import pandas as pd

# Train-test split
from sklearn.model_selection import train_test_split

# Pipeline and preprocessing
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# Handling missing values
from sklearn.impute import SimpleImputer

# Feature scaling and encoding
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Machine Learning model
from sklearn.linear_model import LogisticRegression

# Model evaluation
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

# Save and load the trained pipeline
import joblib

In [2]:
from google.colab import files

uploaded = files.upload()

df = pd.read_csv(next(iter(uploaded)))

Saving Customer_Churn_Dataset_GKeerthana.csv to Customer_Churn_Dataset_GKeerthana.csv


In [3]:
# Step 3: Load and Inspect the Dataset

import pandas as pd

# Load the dataset
df = pd.read_csv("Customer_Churn_Dataset_GKeerthana.csv")

# Display first 5 rows
print("First 5 Rows:")
print(df.head())

# Display dataset information
print("\nDataset Information:")
df.info()

# Display the shape of the dataset
print("\nDataset Shape:")
print(df.shape)

# Display column names
print("\nColumn Names:")
print(df.columns.tolist())

# Check data types
print("\nData Types:")
print(df.dtypes)

# Check missing values
print("\nMissing Values:")
print(df.isnull().sum())

# Check duplicate rows
print("\nDuplicate Rows:")
print(df.duplicated().sum())

# Display statistical summary
print("\nStatistical Summary:")
print(df.describe())

# Display the target column distribution
print("\nTarget Column Distribution:")
print(df["Churn"].value_counts())

First 5 Rows:
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV StreamingMovies       

In [6]:
# Step 4: Remove unnecessary columns

# Remove CustomerID if it exists
if "customerID" in df.columns:
    df = df.drop("customerID", axis=1)
    print("CustomerID column removed.")
else:
    print("CustomerID column not found. Skipping removal.")

# Separate features and target
X = df.drop("Churn", axis=1)
y = df["Churn"]

# Display feature and target shapes
print("\nFeature Matrix Shape:", X.shape)
print("Target Vector Shape:", y.shape)

# Display feature column names
print("\nFeature Columns:")
print(X.columns.tolist())

# Display target values
print("\nTarget Classes:")
print(y.unique())

CustomerID column removed.

Feature Matrix Shape: (7043, 19)
Target Vector Shape: (7043,)

Feature Columns:
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges']

Target Classes:
['No' 'Yes']


In [7]:
# Step 5: Identify Numerical and Categorical Features

# Identify numerical columns
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Identify categorical columns
categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

# Display the columns
print("Numerical Features:")
print(numeric_features)

print("\nCategorical Features:")
print(categorical_features)

Numerical Features:
['SeniorCitizen', 'tenure', 'MonthlyCharges']

Categorical Features:
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'TotalCharges']


In [8]:
# Step 6: Split the Dataset

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Display the shapes of the split datasets
print("Training Features Shape :", X_train.shape)
print("Testing Features Shape  :", X_test.shape)
print("Training Target Shape   :", y_train.shape)
print("Testing Target Shape    :", y_test.shape)

# Check the class distribution
print("\nTraining Target Distribution:")
print(y_train.value_counts())

print("\nTesting Target Distribution:")
print(y_test.value_counts())

Training Features Shape : (5634, 19)
Testing Features Shape  : (1409, 19)
Training Target Shape   : (5634,)
Testing Target Shape    : (1409,)

Training Target Distribution:
Churn
No     4139
Yes    1495
Name: count, dtype: int64

Testing Target Distribution:
Churn
No     1035
Yes     374
Name: count, dtype: int64


In [9]:
# Step 7: Numerical Preprocessing Pipeline

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

numeric_pipeline = Pipeline(steps=[
    ("missing_value_handler", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

print("Numerical Pipeline Created Successfully!")
print(numeric_pipeline)

Numerical Pipeline Created Successfully!
Pipeline(steps=[('missing_value_handler', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())])


In [10]:
# Step 8: Categorical Preprocessing Pipeline

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

categorical_pipeline = Pipeline(steps=[
    ("missing_value_handler", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

print("Categorical Pipeline Created Successfully!")
print(categorical_pipeline)

Categorical Pipeline Created Successfully!
Pipeline(steps=[('missing_value_handler',
                 SimpleImputer(strategy='most_frequent')),
                ('encoder', OneHotEncoder(handle_unknown='ignore'))])


In [11]:
# Step 9: Combine Numerical and Categorical Pipelines

from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ]
)

print("ColumnTransformer Created Successfully!")
print(preprocessor)

ColumnTransformer Created Successfully!
ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('missing_value_handler',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['SeniorCitizen', 'tenure', 'MonthlyCharges']),
                                ('cat',
                                 Pipeline(steps=[('missing_value_handler',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['gender', 'Partner', 'Dependents',
                                  'PhoneService', 'MultipleLines',
                                  'InternetService', 'OnlineSecurity',
                              

In [12]:
# Step 10: Create the Final Machine Learning Pipeline

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

model_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])

print("Final Machine Learning Pipeline Created Successfully!")
print(model_pipeline)

Final Machine Learning Pipeline Created Successfully!
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('missing_value_handler',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['SeniorCitizen', 'tenure',
                                                   'MonthlyCharges']),
                                                 ('cat',
                                                  Pipeline(steps=[('missing_value_handler',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
            

In [13]:
model_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('missing_value_handler',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['SeniorCitizen', 'tenure',
                                                   'MonthlyCharges']),
                                                 ('cat',
                                                  Pipeline(steps=[('missing_value_handler',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['gender', 'Partner',
                                                   'Dependents', 'PhoneService',
                                                   'MultipleLines',
                                                   'InternetService',
                                                   'OnlineSecurity',
                                                   'OnlineBackup',
                                                   'DeviceProtection',
                                                   'TechSupport', 'StreamingTV',
                                                   'StreamingMovies',
                                                   'Contract',
                                                   'PaperlessBilling',
                                                   'PaymentMethod',
                                                   'TotalCharges'])])),
                ('model', LogisticRegression(max_iter=1000, random_state=42))])

In [14]:
# Step 11: Train the Machine Learning Pipeline

model_pipeline.fit(X_train, y_train)

print("Model training completed successfully!")

Model training completed successfully!


In [16]:
# Step 12: Make Predictions

# Predict churn for the test dataset
y_pred = model_pipeline.predict(X_test)

# Display the first 10 predictions
print("First 10 Predictions:")
print(y_pred[:10])

# Display the first 10 actual values
print("\nFirst 10 Actual Values:")
print(y_test.iloc[:10].values)

First 10 Predictions:
['No' 'Yes' 'No' 'No' 'No' 'Yes' 'No' 'No' 'No' 'No']

First 10 Actual Values:
['No' 'No' 'No' 'No' 'No' 'No' 'No' 'No' 'No' 'Yes']


In [17]:
# Step 13: Evaluate the Model

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Calculate Accuracy
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", round(accuracy * 100, 2), "%")

# Display Confusion Matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Display Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 79.42 %

Confusion Matrix:
[[917 118]
 [172 202]]

Classification Report:
              precision    recall  f1-score   support

          No       0.84      0.89      0.86      1035
         Yes       0.63      0.54      0.58       374

    accuracy                           0.79      1409
   macro avg       0.74      0.71      0.72      1409
weighted avg       0.79      0.79      0.79      1409



In [18]:
# Step 14: Save the Trained Pipeline

import joblib

# Save the complete pipeline
joblib.dump(model_pipeline, "customer_churn_pipeline.pkl")

print("Pipeline saved successfully as 'customer_churn_pipeline.pkl'")

Pipeline saved successfully as 'customer_churn_pipeline.pkl'


In [19]:
# Step 15: Load the Saved Pipeline

import joblib

# Load the saved pipeline
loaded_pipeline = joblib.load("customer_churn_pipeline.pkl")

print("Pipeline loaded successfully!")
print(loaded_pipeline)

Pipeline loaded successfully!
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('missing_value_handler',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['SeniorCitizen', 'tenure',
                                                   'MonthlyCharges']),
                                                 ('cat',
                                                  Pipeline(steps=[('missing_value_handler',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                    

In [20]:
print(X.columns.tolist())

['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges']


In [24]:
import pandas as pd

new_customer = pd.DataFrame({
    "gender": ["Female"],          # lowercase
    "SeniorCitizen": [0],
    "Partner": ["Yes"],
    "Dependents": ["No"],
    "tenure": [12],                # lowercase
    "PhoneService": ["Yes"],
    "MultipleLines": ["No"],
    "InternetService": ["Fiber optic"],
    "OnlineSecurity": ["No"],
    "OnlineBackup": ["Yes"],
    "DeviceProtection": ["No"],
    "TechSupport": ["No"],
    "StreamingTV": ["Yes"],
    "StreamingMovies": ["Yes"],
    "Contract": ["Month-to-month"],
    "PaperlessBilling": ["Yes"],
    "PaymentMethod": ["Electronic check"],
    "MonthlyCharges": [85.50],
    "TotalCharges": [1026.00]
})

prediction = loaded_pipeline.predict(new_customer)

print("Churn Prediction:", prediction[0])

Churn Prediction: Yes


In [25]:
# Predict churn probability
probability = loaded_pipeline.predict_proba(new_customer)

print("Probability of NOT Churning: {:.2f}%".format(probability[0][0] * 100))
print("Probability of Churning: {:.2f}%".format(probability[0][1] * 100))

Probability of NOT Churning: 26.33%
Probability of Churning: 73.67%
